# 02 · Spatial exploration

GeoPackage layers, CRS, nodes, renewable installations, buffers and constraints. Open the same file in QGIS: `results/demo/bess_master.gpkg`.

> ⚠️ **These notebooks run on the SYNTHETIC demonstration data.** Every number below is fictional and says nothing about real Catalan grid nodes. To explore real data, change `cfg.mode_paths("demo")` to `cfg.mode_paths("real")` once the real stages have run.

In [ ]:
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd

from bess_catalonia.config import load_config
from bess_catalonia.pipeline import run_pipeline

cfg = load_config()  # finds the project root from the notebook's location
paths = cfg.mode_paths("demo")  # SYNTHETIC demonstration outputs (results/demo/)

# Make sure the demo outputs exist (fast no-op when they are up to date).
if not (paths.processed_root / "integrate" / "_stage.json").exists():
    run_pipeline("demo", run_all=True, root=cfg.root, log_level="WARNING")
print("Project root:", cfg.root)
print("Reading:", paths.results_root)

## 1. Layers in the master GeoPackage

In [ ]:
gpkg = paths.results_root / "bess_master.gpkg"
gpd.list_layers(gpkg)

## 2. CRS
All analysis layers use **EPSG:25831** (ETRS89 / UTM 31N, metres). Lon/lat (EPSG:4326) is kept as columns for exchange.

In [ ]:
nodes = gpd.read_file(gpkg, layer="nodes_integrated")
print(nodes.crs.name, "| projected:", nodes.crs.is_projected, "| units:", nodes.crs.axis_info[0].unit_name)
nodes[["node_id", "lon", "lat"]].assign(x=nodes.geometry.x.round(), y=nodes.geometry.y.round()).head()

## 3. Grid nodes and renewable installations

In [ ]:
inst = gpd.read_file(gpkg, layer="renewable_installations")
fig, ax = plt.subplots(figsize=(7, 7))
for cat, color in [("operating", "#27ae60"), ("pipeline", "#f39c12"), ("unknown", "grey")]:
    sub = inst[inst["status_category"] == cat]
    sub.plot(ax=ax, markersize=sub["capacity_mw"].fillna(1) / 2, color=color, alpha=0.6,
             label=f"{cat} ({len(sub)})")
nodes.plot(ax=ax, color="black", marker="^", markersize=50, label="grid node")
ax.legend(); ax.set_title("Nodes and renewable installations (SYNTHETIC)")

## 4. Buffers and a spatial relationship
Re-compute operating capacity within 10 km with a GeoPandas spatial join, and check it agrees with the pipeline's `renewable_mw_10km`. Like the pipeline, drop records whose source id is a duplicate (`duplicate_source_id`), otherwise their capacity is counted twice.

In [ ]:
from bess_catalonia.gis.spatial import make_buffers

buffers = make_buffers(nodes, 10)
operating = inst[(inst["status_category"] == "operating") & ~inst["duplicate_source_id"].astype(bool)]
joined = gpd.sjoin(operating, buffers, predicate="within")
check = joined.groupby("node_id")["capacity_mw"].sum().reindex(nodes["node_id"], fill_value=0)
compare = nodes.set_index("node_id")["renewable_mw_10km"].to_frame("pipeline").assign(notebook=check)
print("max abs difference (MW):", (compare.pipeline - compare.notebook).abs().max().round(6))
compare.head()

> Points exactly on a buffer boundary can differ between `within` and the pipeline's distance test; any difference should be ~0.

## 5. Geographic constraints and suitable land

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))
gpd.read_file(gpkg, layer="suitable_land_net").plot(ax=ax, color="#9bd17f", alpha=0.7)
gpd.read_file(gpkg, layer="hard_constraints").plot(ax=ax, facecolor="none", edgecolor="#c0392b", hatch="//")
buffers.boundary.plot(ax=ax, color="grey", linewidth=0.5)
nodes.plot(ax=ax, column="suitable_land_ha_5km", cmap="Greens", markersize=60, edgecolor="black", legend=True)
ax.set_title("Suitable land (net of hard constraints) and 10 km buffers (SYNTHETIC)")

In [ ]:
nodes[["node_id", "suitable_land_ha_5km", "protected_area_conflict", "flood_risk_at_node",
       "distance_to_major_road_km"]].sort_values("suitable_land_ha_5km", ascending=False).head(10)